<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-04-rag/lesson-4.4-search-grounding/practice/GCP_Capstone_4.4_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 4.4 — Vertex AI Search & Google Search Grounding

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup

Run this cell first. It installs the SDKs, authenticates with Application Default Credentials, and initializes the `google-genai` client against Vertex AI. Every exercise below depends on the names defined here (`client`, `PROJECT_ID`, `LOCATION`, `types`, `discoveryengine`).

In [ ]:
# Colab may print pip dependency-conflict warnings for ydf and
# google-ai-generativelanguage (both Colab pre-installs this lesson does NOT use) —
# they are harmless. If a later cell raises a protobuf error, do
# Runtime > Restart session and re-run (packages are already installed).
!pip install -q google-genai google-cloud-discoveryengine
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE
LOCATION = 'global'  # Discovery Engine data stores live in 'global'

from google import genai
from google.genai import types
from google.cloud import discoveryengine_v1 as discoveryengine

# Vertex AI client (google-genai unified SDK). NEVER vertexai=True, never vertexai.generative_models.
client = genai.Client(enterprise=True, project=PROJECT_ID, location='global')  # Gemini 3.x generation: global

USD_INR = 85  # cost display conversion

## Exercise 1: Google Search Grounding

**Difficulty:** Easy

Enable the `google_search` tool. Ask a current-events question. Print the grounded answer.

1. Use `types.Tool(google_search=types.GoogleSearch())`
2. Ask about recent news
3. Print `response.text`

In [ ]:
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='What are the latest developments in India semiconductor policy?',
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())]
    ),
)
print(response.text[:500])
# Expected: a grounded answer with current information not in training data.

## Exercise 2: Extract Grounding Metadata

**Difficulty:** Easy

Print `web_search_queries`, `grounding_chunks` (source URIs), and `grounding_supports` (text-to-source mappings).

1. Access `response.candidates[0].grounding_metadata`
2. Print `web_search_queries`
3. Loop `grounding_chunks` and `grounding_supports`

In [ ]:
gm = response.candidates[0].grounding_metadata

# What the model searched for
print(f'Search queries: {gm.web_search_queries}')

# Source URIs
if gm.grounding_chunks:
    for chunk in gm.grounding_chunks:
        print(f'  Source: {chunk.web.title}')
        print(f'  URI: {chunk.web.uri}')

# Text-to-source mappings
if gm.grounding_supports:
    for support in gm.grounding_supports:
        print(f'  Claim: {support.segment.text[:80]}...')
        print(f'  Backed by: {support.grounding_chunk_indices}')

## Exercise 3: Render Search Widget

**Difficulty:** Easy

Extract `search_entry_point.rendered_content` and display the HTML widget in Colab. (Rendering this widget is required by the Google Search grounding Terms of Service.)

1. Access `gm.search_entry_point.rendered_content`
2. Display with `IPython.display.HTML()`
3. Verify Google branding appears

In [ ]:
from IPython.display import HTML, display

if gm.search_entry_point:
    display(HTML(gm.search_entry_point.rendered_content))
else:
    print('No search widget in this response')
# Expected: the Google Search widget renders with suggested queries.

## Exercise 4: Create Vertex AI Search Data Store

**Difficulty:** Medium

Create a data store with the `discoveryengine` SDK, then import documents from GCS.

1. Create a `DataStoreServiceClient`
2. Call `create_data_store()`
3. Import documents with `import_documents()`

First enable the Discovery Engine API (shell), then create the data store, then import.

In [ ]:
%%bash
gcloud services enable discoveryengine.googleapis.com

In [ ]:
# --- Create the data store ---
ds_client = discoveryengine.DataStoreServiceClient()
parent = ds_client.collection_path(PROJECT_ID, LOCATION, 'default_collection')

data_store = discoveryengine.DataStore(
    display_name='DocuMind KB',
    industry_vertical=discoveryengine.IndustryVertical.GENERIC,
    solution_types=[discoveryengine.SolutionType.SOLUTION_TYPE_SEARCH],
    content_config=discoveryengine.DataStore.ContentConfig.CONTENT_REQUIRED)

try:
    op = ds_client.create_data_store(
        request=discoveryengine.CreateDataStoreRequest(
            parent=parent, data_store_id='documind-kb-test', data_store=data_store))
    result = op.result()
    print(f'Data store: {result.name}')
except Exception as e:
    print(f'Error (may already exist): {e}')

In [ ]:
# --- Import documents from GCS (INCREMENTAL mode) ---
doc_client = discoveryengine.DocumentServiceClient()
branch = (f'projects/{PROJECT_ID}/locations/{LOCATION}'
          f'/collections/default_collection/dataStores/documind-kb-test'
          f'/branches/default_branch')

try:
    import_op = doc_client.import_documents(
        request=discoveryengine.ImportDocumentsRequest(
            parent=branch,
            gcs_source=discoveryengine.GcsSource(
                input_uris=['gs://YOUR-BUCKET/docs/*'],  # CHANGE
                data_schema='content'),
            reconciliation_mode=discoveryengine.ImportDocumentsRequest
                .ReconciliationMode.INCREMENTAL))
    result = import_op.result(timeout=600)
    print('Documents imported')
except Exception as e:
    print(f'Error: {e}')

## Exercise 5: Search with Summaries

**Difficulty:** Medium

Execute a search with a `summary_spec` and extractive answers. Print the summary plus citations.

1. Build a `SearchRequest` with a `ContentSearchSpec`
2. Include `summary_spec` with `include_citations=True`
3. Print `summary_text` and result titles

In [ ]:
search_client = discoveryengine.SearchServiceClient()
serving_config = search_client.serving_config_path(
    PROJECT_ID, LOCATION, 'documind-kb-test', 'default_config')

try:
    request = discoveryengine.SearchRequest(
        serving_config=serving_config,
        query='What is RAG?',
        page_size=5,
        content_search_spec=discoveryengine.SearchRequest.ContentSearchSpec(
            snippet_spec=discoveryengine.SearchRequest.ContentSearchSpec
                .SnippetSpec(return_snippet=True),
            summary_spec=discoveryengine.SearchRequest.ContentSearchSpec
                .SummarySpec(summary_result_count=3, include_citations=True)))
    resp = search_client.search(request)
    if resp.summary:
        print(f'Summary: {resp.summary.summary_text}')
    for r in resp.results:
        doc = r.document.derived_struct_data
        print(f'  Title: {doc.get("title")}')
except Exception as e:
    print(f'Error: {e}')
# Expected: an LLM summary carrying [1], [2] citations, followed by the search result titles.

## Exercise 6: Cost Comparison Calculator

**Difficulty:** Medium

Build a function comparing monthly cost across all 4 RAG approaches at various volumes.

1. Calculate costs for DIY, RAG Engine, Search, and Google Search
2. Include free tiers and base costs
3. Print the comparison at 10K and 100K queries

In [ ]:
def compare_costs(queries_per_month):
    q = queries_per_month
    costs = {
        'DIY RAG (Firestore)': q * 0.0003 + 0,  # ~$0.0003/query, no infra
        'RAG Engine': 65 + q * 0.0025,  # $65 base + $2.50/1K grounding
        'Vertex AI Search (Standard)': q * 0.0015 + 5,  # $1.50/1K + storage
        'Vertex AI Search (Enterprise)': q * 0.004 + 5,  # $4/1K + storage
        'Google Search (2.5)': max(0, q - 15000) * 0.035 / 1,  # 500/day free
        'Google Search (3.x)': max(0, q - 5000) * 0.014 / 1,  # 5K/mo free
    }
    print(f'Monthly cost comparison at {q:,} queries/month:')
    for name, cost in sorted(costs.items(), key=lambda x: x[1]):
        inr = cost * USD_INR
        print(f'  {name}: ${cost:.2f} (Rs {inr:.0f})')

compare_costs(10000)
print()
compare_costs(100000)

## Exercise 7: Search as RAG Engine Backend

**Difficulty:** Challenge

Create a RAG Engine corpus backed by Vertex AI Search, query it via RAG Engine, and compare with a direct search.

1. Create a corpus with `vertex_ai_search` in the `backend_config`
2. Query via RAG Engine
3. Compare the results with the direct search from Exercise 5

RAG Engine can point its retrieval at an existing Vertex AI Search data store instead of managing its own vector store. You wire the data store resource name into the corpus's `vertex_ai_search_config`, then let the model retrieve through it with the `VertexRagStore` tool.

In [ ]:
# RAG Engine corpus whose retrieval backend is the Vertex AI Search data store
# created in Exercise 4. RAG Engine corpora live in a regional location.
import vertexai
from vertexai import rag

vertexai.init(project=PROJECT_ID, location='us-central1')

# Full resource name of the Search data store (from Exercise 4)
DATA_STORE = (f'projects/{PROJECT_ID}/locations/{LOCATION}'
              f'/collections/default_collection/dataStores/documind-kb-test')

try:
    corpus = rag.create_corpus(
        display_name='documind-search-backed',
        vertex_ai_search_config=rag.VertexAiSearchConfig(
            serving_config=f'{DATA_STORE}/servingConfigs/default_search'))
    print(f'Corpus: {corpus.name}')
except Exception as e:
    print(f'Error creating corpus (may already exist): {e}')
    corpus = None

In [ ]:
# Query THROUGH RAG Engine: the model retrieves from the Search-backed corpus,
# then grounds its answer on what came back.
if corpus:
    rag_tool = types.Tool(retrieval=types.Retrieval(
        vertex_rag_store=types.VertexRagStore(
            rag_resources=[types.VertexRagStoreRagResource(rag_corpus=corpus.name)])))
    try:
        rag_resp = client.models.generate_content(
            model='gemini-3.6-flash',
            contents='What is RAG?',
            config=types.GenerateContentConfig(tools=[rag_tool]))
        print('--- RAG Engine (Search-backed) ---')
        print(rag_resp.text[:400])
    except Exception as e:
        print(f'RAG Engine query error: {e}')

# Compare: same question, straight to Vertex AI Search (from Exercise 5).
try:
    direct = search_client.search(discoveryengine.SearchRequest(
        serving_config=serving_config, query='What is RAG?', page_size=3))
    print('\n--- Direct Vertex AI Search ---')
    for r in direct.results:
        print(f'  {r.document.derived_struct_data.get("title")}')
except Exception as e:
    print(f'Direct search error: {e}')
# Expected: RAG Engine reuses Search's retrieval — same underlying quality,
# with more orchestration flexibility around it.

## Exercise 8: GroundedSearch Module

**Difficulty:** Challenge

Build a `GroundedSearch` class with `search()`, citation extraction, widget rendering, and cost tracking.

1. Implement `search()` with the `google_search` tool
2. Extract citations from `grounding_metadata`
3. Track query count and cost

In [ ]:
class GroundedSearch:
    def __init__(self, project, location='global'):  # generation-only client -> global
        self.client = genai.Client(enterprise=True, project=project, location=location)
        self.query_count = 0
        self.total_cost = 0.0

    def search(self, question, model='gemini-3.6-flash'):
        response = self.client.models.generate_content(
            model=model, contents=question,
            config=types.GenerateContentConfig(
                tools=[types.Tool(google_search=types.GoogleSearch())]))
        self.query_count += 1
        self.total_cost += 0.014
        citations = []
        gm = response.candidates[0].grounding_metadata
        if gm and gm.grounding_chunks:
            for chunk in gm.grounding_chunks:
                citations.append({'title': chunk.web.title, 'uri': chunk.web.uri})
        return {
            'answer': response.text,
            'citations': citations,
            'queries': gm.web_search_queries if gm else [],
            'widget': gm.search_entry_point.rendered_content if gm and gm.search_entry_point else ''}

    def report(self):
        print(f'Queries: {self.query_count} | Cost: ${self.total_cost:.2f}')

gs = GroundedSearch(PROJECT_ID)
r = gs.search('Latest AI developments in India 2026')
print(f'Answer: {r["answer"][:200]}...')
print(f'Citations: {len(r["citations"])}')
for c in r['citations'][:3]:
    print(f'  {c["title"]}: {c["uri"]}')
gs.report()